# Evaluate Spanish product translations

[Open in Colab](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/use-cases/translation-eval.ipynb)

A translation can sound natural and still break a placeholder or overflow a button. Score six English-to-Spanish strings for meaning and register; check terminology, placeholders and a character budget without an LLM.

## Install and authenticate

Start in a fresh Python 3.12 kernel. OpenAI translates and grades register; Future AGI scores translation accuracy.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                'agent-learning-kit==0.3.7', 'openai==2.54.0'], check=True)

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value.strip():
            raise ValueError(f"{name} is required.")
        os.environ[name] = value.strip()
        del value

## Define a source and a reference for each surface

A reference illustrates the expected tone, not the only valid translation. Only the two fixed-width UI surfaces get a character budget.

In [ ]:
cases = [
    ("button", "Save changes", "Guardar cambios", True),
    ("error", "Your API key is invalid. Please try again.", "Su clave de API no es válida. Inténtelo de nuevo.", False),
    ("marketing", "Find your next trail, {name}!", "¡Encuentra tu próxima ruta, {name}!", False),
    ("technical", "To enable SSO, paste your SAML metadata URL.", "Para habilitar SSO, pegue la URL de metadatos SAML.", False),
    ("tooltip", "Show order details", "Ver detalles del pedido", True),
    ("legal", "By continuing, you accept the return policy.", "Al continuar, acepta la política de devoluciones.", False),
]

## Generate Spanish translations

In [ ]:
import json
from fi.evals import evaluate, Evaluator
from openai import OpenAI

client = OpenAI()
reader = Evaluator()


def hosted(metric, **inputs):
    result = evaluate(metric, model="turing_small", **inputs)
    if result.status != "completed" or result.score is None:
        raise RuntimeError(f"{metric} returned no score: {result.error or result.reason}")
    eval_id = result.metadata["eval_id"]
    record = reader.get_eval_result(eval_id)["result"]["result"]
    if str(record["eval_id"]) != str(eval_id) or record["name"] != metric or record.get("error_message"):
        raise RuntimeError(f"Could not confirm stored {metric} result.")
    return result


def grade(rubric, **inputs):
    result = evaluate(engine="llm", model="openai/gpt-4o-mini", prompt=rubric, **inputs)
    if result.status != "completed" or result.score is None:
        raise RuntimeError("The rubric judge did not return a score.")
    # The kit serializes a successful custom judgment in reason. A provider or
    # parse failure is not an ordinary low-quality result.
    try:
        detail = json.loads(result.reason)
        if float(detail["score"]) != result.score or not detail["reason"].strip():
            raise ValueError
    except (ValueError, TypeError, KeyError) as exc:
        raise RuntimeError("The rubric judge did not return a usable verdict.") from exc
    return result.score, detail["reason"]

In [ ]:
def translate(kind, source, compact):
    rules = ("Translate the English product string into Spanish. Use formal or neutral Spanish for "
             "UI, errors, help and legal copy. Informal address is fine for marketing. Keep API, SSO, "
             "SAML and URL unchanged. Preserve placeholders exactly. Return only the translation.")
    if compact:
        rules += f" The UI budget is {len(source) * 13 // 10} characters, including spaces."
    resp = client.chat.completions.create(model="gpt-4o-mini", temperature=0,
        messages=[{"role": "system", "content": rules},
                  {"role": "user", "content": f"Surface: {kind}\nSource: {source}"}])
    text = (resp.choices[0].message.content or "").strip()
    if not text:
        raise RuntimeError(f"Empty translation for {kind}")
    return text


rows = []
for kind, source, reference, compact in cases:
    translation = translate(kind, source, compact)
    rows.append(dict(kind=kind, source=source, reference=reference, translation=translation, compact=compact))
    print(f"{kind}: {source} → {translation}")

## Score meaning and register independently

The hosted accuracy metric compares the English source and Spanish translation. The second judge checks the surface-specific tone. A natural translation need not match the reference word for word.

In [ ]:
tone_rubric = ("Score 1 for register appropriate to the surface, otherwise 0. Marketing permits "
               "informal Spanish. Other surfaces require formal or neutral Spanish (including an "
               "infinitive button label). Use the reference as guidance, not an exact string match. "
               "Explain briefly.")
for row in rows:
    accuracy = hosted("translation_accuracy", input=row["source"], output=row["translation"])
    register, why = grade(tone_rubric, surface=row["kind"], source=row["source"],
                          reference=row["reference"], translation=row["translation"])
    row.update(accuracy=accuracy.score, accuracy_reason=accuracy.reason,
               eval_id=accuracy.metadata["eval_id"], register=register, register_reason=why)

## Check the product constraints

Count each required term and placeholder, not just whether it appears. These strings use Python-style fields such as `{name}`; `{{name}}` is literal text. The check also recognizes `%s`. The length budget counts characters, not pixels. Use layout tests for the actual component.

In [ ]:
import re
from collections import Counter
from string import Formatter

term = re.compile(r"\b(?:API|SSO|SAML|URL)\b")


def placeholders(text):
    # Escaped braces are literal text. Keep each field's formatting and conversion.
    fields = Counter((name, spec, conv) for _, name, spec, conv in Formatter().parse(text)
                     if name is not None)
    fields["%s"] = re.findall(r"%%|%s", text).count("%s")
    return fields


for row in rows:
    src, dst = row["source"], row["translation"]
    budget = len(src) * 13 // 10 if row["compact"] else None
    try:
        placeholders_ok = placeholders(src) == placeholders(dst)
    except ValueError:  # An unbalanced brace makes the template unusable.
        placeholders_ok = False
    row.update(terms_ok=Counter(term.findall(src)) == Counter(term.findall(dst)),
               placeholders_ok=placeholders_ok,
               budget=budget, length_ok=budget is None or len(dst) <= budget)

## Build a human review queue

Require at least 0.8 for meaning and full marks for register. Treat a model score as a triage signal, not permission to publish.

In [ ]:
import csv

for row in rows:
    checks = {"meaning": row["accuracy"] >= 0.8, "register": row["register"] == 1,
              "terms": row["terms_ok"], "placeholders": row["placeholders_ok"], "length": row["length_ok"]}
    row["flags"] = ", ".join(k for k, ok in checks.items() if not ok)
    print(f"{row['kind']:10} {row['translation']} | {row['flags'] or 'no checks flagged'}")
    if not checks["meaning"]:
        print("  Accuracy:", row["accuracy_reason"])
    if not checks["register"]:
        print("  Register:", row["register_reason"])
    if not checks["length"]:
        print(f"  Length: {len(row['translation'])} characters; budget {row['budget']}")

with open("translation_scores.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

The CSV preserves the source, candidate, reference, model judgments, hosted evaluation ID and structural checks for each string. Send flagged rows to a bilingual reviewer and sample unflagged rows to look for misses.